# 第6回 分類モデル（線形判別分析、ロジスティック回帰）を実装する（解答例・教員用）


---


## 前回からの接続


第1〜5回は、利用台数・1泊の料金・住宅価格と、ずっと数を当ててきました。第5回では係数が大きくなりすぎないように抑え、回帰の道具はひととおりそろいました。

今日から問いの形が変わります。当てる相手は数ではなく区分です。「心臓病あり／なし」の二択で、データも住宅から医療に移ります。そして、二種類の間違い（病気を見逃す、健康な人を病気と言う）は、重み（weight）が違います。正解率という一つの数字では見えないものを、混同行列（confusion matrix）で確かめていきます。


---


## 今日の分析目標


あなたは健診センターの医師です。検査の結果から、心臓病の疑いがある人を見つけ出し、何人を見逃すかを数で言えるようにする。

心臓病かどうかを確率で当てるロジスティック回帰を立て、その判定がどう当たり、どう外れたかを、混同行列のマスを自分で数えて確かめます。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは健診センターの医師です。胸の痛みなどで来た人に、いくつかの検査をします。心臓カテーテルの精密検査は体への負担が大きいので、気軽には回せません。

決めたいのは、検査の結果から、精密検査に回す人を選ぶことです。そのとき見逃す人が何人出るかも、前もって知っておきたいところです。病気の人を健康だとして帰す見逃しは、治療の機会を失わせます。健康な人を病気と言う誤検出は、再検査で済みます。2つの間違いは、重さが違います。


### 手元のデータ

米国クリーブランドの病院で、心臓の検査を受けた303人の記録です（UCI Machine Learning RepositoryのHeart Disease）。1行が1人で、年齢・性別・血圧・コレステロール・最大心拍数など、13の検査値が並びます。

答えの列`target`は、心臓カテーテルの検査で、太い血管の少なくとも1本が半分以上狭まっていた人を1（心臓病あり）、そうでない人を0としています。


### 最後に出す答え

「確率がいくつ以上なら精密検査に回すか」という判定のしかたと、その判定で何人を見つけ、何人を見逃すかの人数です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 0/1を直線で当てるとうまくいかない理由と、ロジスティック回帰が確率を出すしくみを説明できる
- 確率をしきい値で判定に変え、配列の比較とTrue/Falseの集計で正解率を計算できる
- 混同行列の4つのマスを、条件を重ねたTrue/Falseの集計で自分で数えられる
- 適合率（precision）と再現率（recall）を式から計算し、医療で重い「見逃し」を数で言える
- ROC曲線とAUCで、しきい値によらない判定の良さを比べられる


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出す
- 入力：なし
- 出力：「セットアップ完了」の表示


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import roc_curve, roc_auc_score

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ② の解析と、③ の解釈が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (2, 3)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. どんな人たちのデータか


見つけ出す前に、まず、どんな人たちのデータかを見ます。クリーブランドの病院で心臓の検査を受けた303人のデータです（UCI）。1行が1人で、`target`が「心臓病あり(1)／なし(0)」です。


### お手本：読み込んで、2つのグループを比べる

- 役割：データを読み込み、心臓病あり・なしの人数と、主な検査値の平均を比べる
- 入力：データの置き場所とファイル名
- 出力：表`df`、人数、グループごとの平均


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
df = pd.read_csv(f'{DATA_DIR}/heart.csv')
print((df['target'] == 1).sum(), '人が心臓病あり /', len(df), '人')
cols = ['age', 'thalach', 'oldpeak']
print(df.groupby('target')[cols].mean().round(1))


303人のうち、心臓病ありが139人で、ほぼ半々です。列は、年齢（`age`）、性別（`sex`、1が男性）、胸の痛みの型（`cp`）、安静時の血圧（`trestbps`）、コレステロール（`chol`）、運動したときの最大心拍数（`thalach`）、運動で起きる胸の痛み（`exang`）、運動で心電図が下がる幅（`oldpeak`）など、13本です。

心臓病ありの人は、平均すると年齢がやや高く（56.6歳と52.6歳）、最大心拍数が低く（139と158）、心電図の下がり幅が大きい（1.6と0.6）ことがわかります。ただし、どれか一つの列だけで、はっきり分けられるわけではありません。最大心拍数を、グループ別に重ねてみます。


In [ ]:
# @title 図：最大心拍数の分布（グループ別）
for t, c, lab in [(0, '#0066cc', '健康'), (1, '#e63946', '心臓病')]:
    plt.hist(df.loc[df['target'] == t, 'thalach'], bins=20,
             alpha=0.5, color=c, label=lab)
plt.xlabel('最大心拍数（thalach）'); plt.ylabel('人数'); plt.legend()
plt.show()


心臓病の人（赤）は左に寄っていますが、2つの山は大きく重なっています。最大心拍数だけでは、分けきれません。13本の列をまとめて使います。


### おさらい：def

```python
def 名前(引数):
    return 戻り値
```

`def`の後ろが関数の名前と引数（入力）、`return`の後ろが戻り値（出力）でした（第3回）。第3回では、補う値と標準化（standardization）の物差しを訓練で決めて、表を整える関数`prep`を作りました。今日も同じものを使いますが、使う列の並びも、3つ目の引数`cols`で渡す形にします。


### 変えてみる：訓練とテストに分け、第3回のprepで整える

`stratify=df['target']`を付けると、訓練とテストで、心臓病ありの割合がそろうように分けられます。`prep`は、第3回と同じく、補う値（中央値）と標準化の物差しを`base`で決めます。違うのは、使う列を引数`cols`で受け取るところだけです。


In [ ]:
feat = [c for c in df.columns if c != 'target']      # 13本の検査値
tr, te = train_test_split(df, test_size=0.2, random_state=42,
                          stratify=df['target'])
y_tr, y_te = tr['target'].values, te['target'].values

def prep(base, df_in, cols):  # 物差しは base で決める（第3回）
    med = base[cols].median()
    Xb = base[cols].fillna(med).values
    X = df_in[cols].fillna(med).values
    return (X - Xb.mean(axis=0)) / Xb.std(axis=0)

Z_tr, Z_te = prep(tr, tr, feat), prep(tr, te, feat)
print(len(y_tr), len(y_te))


---


## 2. 確率で当てる：ロジスティック回帰


1本の列では、心臓病あり・なしの山が重なりました。13本の列をまとめて当てるには、どうすればよいでしょうか。

当てたいのは0/1の二択です。これまでの線形回帰（linear regression）で0/1を当ててはいけないのでしょうか。試すと、うまくいかない理由がはっきり出ます。


In [ ]:
lin = LinearRegression().fit(Z_tr, y_tr)
p_lin = lin.predict(Z_te)
print(p_lin.min().round(2), p_lin.max().round(2))


出てくる値は最小 −0.18・最大1.34まで散らばり、61人中9人が0〜1の外にはみ出します。確率が「マイナス」や「1超え」では、意味をなしません。

ロジスティック回帰（logistic regression）は、線形回帰と同じ「係数 × 列の値の合計」を計算したあと、それをシグモイド関数で0〜1の間に収めて、確率にします。

$$
p = \frac{1}{1 + e^{-z}}, \qquad z = \beta_0 + \sum_j \beta_j x_j
$$

$z$ がどんなに大きくても小さくても、$p$ は必ず0と1の間に入ります。


### お手本：心臓病である確率を出す

- 役割：ロジスティック回帰を訓練で学び、テストの一人ひとりの「心臓病である確率」を出す
- 入力：`Z_tr`・`y_tr`・`Z_te`
- 出力：確率の並び`prob`

`predict_proba`は、「0である確率」と「1である確率」の2列を返すメソッドです。`[:, 1]`で2列目（心臓病である確率）だけを取り出します。

なお、scikit-learnの`LogisticRegression`には、初めから係数の二乗の罰金（L2）がかかっています。前回のRidgeと同じ罰金です。強さは引数`C`で決まり、`C`が小さいほど罰金が強くなります（初めの値は`C=1.0`）。


In [ ]:
clf = LogisticRegression(max_iter=5000).fit(Z_tr, y_tr)
prob = clf.predict_proba(Z_te)[:, 1]      # 心臓病である確率
print(prob[:5].round(2))


テストの先頭5人の確率は、0.22, 0.71, 0.06, 0.04, 0.54です。どれも0〜1に収まっています。


### 変えてみる：LDAでも確率を出す

線形判別分析（linear discriminant analysis）（LDA）は、「病気の人」と「健康な人」の検査値の散らばりから、2つを分ける方向を求める、もう一つの分類（classification）の方法です。使い方は同じです。


In [ ]:
lda = LinearDiscriminantAnalysis().fit(Z_tr, y_tr)
prob_lda = lda.predict_proba(Z_te)[:, 1]
print(prob_lda[:5].round(2))


先頭5人の確率は、ロジスティック回帰と少しずつ違いますが、高い人・低い人の顔ぶれはほぼ同じです。


---


## 3. 確率を、判定に変える


確率のままでは、「精密検査に回すか、回さないか」は決まりません。確率が0.5を超えたら心臓病と判定する、と決めてみます。この区切りの値をしきい値と呼びます。


### おさらい：条件で絞る

`df[ ]`にTrue/Falseの並びを入れると、Trueの行だけが残ります。

```python
df[df['price'] <= 100000]      # 10万円以下の部屋だけ
```

`<=`や`>`で比べると、1つずつTrue/Falseが並ぶのでした（第1回・第3回）。今日は、このTrue/Falseの並びそのものを使います。


### お手本：比べて、判定と正解率を出す

- 役割：確率をしきい値0.5と比べて判定（0/1）にし、正解率を出す
- 入力：確率`prob`と、本当の答え`y_te`
- 出力：判定の並び`pred`と、正解率

`prob > 0.5`は、61人それぞれについて「0.5を超えるか」をTrue/Falseで並べたものです（第1回の「条件で絞る」で使った並びと同じ）。`.astype(int)`でTrueを1、Falseを0に変えます。`pred == y_te`は「判定と答えが同じか」のTrue/Falseの並びで、その`.mean()`はTrueの割合、つまり正解率です（Trueを1、Falseを0として平均するから）。


In [ ]:
pred = (prob > 0.5).astype(int)           # 0.5 を超えたら 1（心臓病）
print(pred[:5])
print((pred == y_te).mean().round(3))      # 正解率


正解率は0.869。61人中53人を当てました。


---


## 4. 混同行列：4つのマスを自分で数える


では、外した8人は、どう外したのでしょうか。病気の人を「健康」と帰したのか、健康な人を「病気」と言ったのか。医療では、この2つの重さがまるで違います。

判定と本当の答えの組み合わせは、4通りあります。

| | 判定：健康（0） | 判定：心臓病（1） |
|---|---|---|
| 本当は健康（0） | 真陰性TN（正しく健康） | 偽陽性FP（誤検出） |
| 本当は心臓病（1） | 偽陰性FN（見逃し） | 真陽性TP（正しく発見） |

この表を混同行列と呼びます。まず、自分で数えます。


### お手本：真陽性（TP）を数える

- 役割：「判定が1」かつ「本当も1」の人数を数える
- 入力：`pred`・`y_te`
- 出力：人数`tp`

`&`は「かつ」です（第1回）。2つのTrue/Falseの並びを`&`でつなぐと、両方Trueの人だけがTrueになります。`.sum()`でTrueの数を数えます。


In [ ]:
tp = ((pred == 1) & (y_te == 1)).sum()    # 判定も本当も心臓病
print(tp)


### 変えてみる：真陰性（TN）を数える

`1`を`0`に変えるだけです。


In [ ]:
tn = ((pred == 0) & (y_te == 0)).sum()    # 判定も本当も健康
print(tn)


### TODO①：誤検出（FP）と見逃し（FN）を数え、答え合わせする

お手本のセルの`==`の右側の0/1を組み替えて、誤検出`fp`（判定は1、本当は0）と見逃し`fn`（判定は0、本当は1）を数えてください。そして、`confusion_matrix(y_te, pred)`と一致するか確かめてください。


In [ ]:
# TODO: fp と fn を数え、confusion_matrix(y_te, pred) と比べる

# 解答例①：誤検出と見逃し
fp = ((pred == 1) & (y_te == 0)).sum()    # 判定は心臓病、本当は健康
fn = ((pred == 0) & (y_te == 1)).sum()    # 判定は健康、本当は心臓病
print('TP', tp, ' TN', tn, ' FP', fp, ' FN', fn)
print(confusion_matrix(y_te, pred))


TP=26、TN=27、FP=6、FN=2。`confusion_matrix`は、左上からTN・FP、下の段がFN・TPの順に並べた表を返します。誤検出は6人、見逃しは2人でした。


In [ ]:
# @title 図：混同行列
cm = confusion_matrix(y_te, pred)
fig, ax = plt.subplots(figsize=(4.5, 3.8))
ax.imshow(cm, cmap='Blues')
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, v, ha='center', va='center', fontsize=18,
            color='white' if v > cm.max() / 2 else '#1a1a2e')
ax.set_xticks([0, 1], ['健康', '心臓病']); ax.set_yticks([0, 1], ['健康', '心臓病'])
ax.set_xlabel('判定'); ax.set_ylabel('本当'); plt.tight_layout(); plt.show()


---


## 5. 適合率と再現率


4つのマスが数えられれば、目的に合わせた物差しが作れます。

$$
\text{適合率} = \frac{TP}{TP + FP}, \qquad \text{再現率} = \frac{TP}{TP + FN}
$$

- 適合率：心臓病と判定した人のうち、本当に心臓病だった割合。誤検出の少なさ
- 再現率：本当に心臓病の人のうち、見つけられた割合。見逃しの少なさ

健診で重いのは、見逃しです。見逃された人は、治療の機会を失います。


### TODO②：適合率と再現率を、式から計算する

4節で数えた`tp`・`fp`・`fn`から、適合率と再現率を計算して表示してください。そして、`classification_report`の心臓病（1）の行と一致するか確かめてください。


In [ ]:
# TODO: tp, fp, fn から適合率と再現率を計算する

# 解答例②：適合率と再現率
print('適合率:', round(tp / (tp + fp), 2))
print('再現率:', round(tp / (tp + fn), 2))
print(classification_report(y_te, pred))


適合率は26 ÷ 32 ＝ 0.81、再現率は26 ÷ 28 ＝ 0.93です。心臓病の人28人のうち26人を見つけ、2人を見逃しました。

正解率0.869は、この2つの違いを一つの数にまとめて、見えなくしてしまいます。たとえば、病気の人が100人中1人しかいないデータなら、「全員健康」と判定するだけで正解率は0.99になりますが、再現率は0です。正解率だけで判定の良さを語ってはいけないのです。


---


## 6. しきい値によらない良さ：ROCとAUC


5節の適合率と再現率は、3節で決めたしきい値0.5で判定して数えた値です。しきい値を下げれば見逃しは減り、誤検出は増えます。では、しきい値を決める前に、2つの方法（ロジスティック回帰とLDA）のどちらが良いかを比べられないでしょうか。

そのために、2つの割合を使います。

$$
\text{誤検出率} = \frac{FP}{FP + TN}, \qquad \text{検出率} = \frac{TP}{TP + FN}
$$

- 誤検出率：本当は健康な人（混同行列の上の段）のうち、誤って心臓病と判定した割合。しきい値0.5では6 ÷ 33 ＝ 0.18
- 検出率：本当に心臓病の人（下の段）のうち、見つけた割合。再現率と同じで、しきい値0.5では26 ÷ 28 ＝ 0.93

ROC曲線は、しきい値を1から0まで全部試したときの、誤検出率と検出率の組を線でつないだ図です。左上に近いほど、誤検出が少ないまま、よく見つける判定です。曲線の下の面積をAUCと呼び、1に近いほど良い判定です。


### お手本：ロジスティック回帰のROC曲線とAUC

- 役割：確率`prob`からROC曲線を描き、AUCを出す
- 入力：`y_te`・`prob`
- 出力：図とAUC


In [ ]:
fpr, tpr, _ = roc_curve(y_te, prob)
print(round(roc_auc_score(y_te, prob), 3))
plt.plot(fpr, tpr, lw=2.5, label='ロジスティック回帰')
plt.plot([0, 1], [0, 1], '--', color='gray')      # でたらめな判定
plt.xlabel('誤検出率'); plt.ylabel('検出率（再現率）'); plt.legend()
plt.show()


AUCは0.951。曲線は左上にぐっと寄っていて、よく分けられています。点線は、でたらめに判定したときの線（AUC 0.5）です。


### TODO③：LDAのAUCと比べる

2節のLDAの確率`prob_lda`で、同じようにAUCを計算し、ロジスティック回帰と比べてください。


In [ ]:
# TODO: roc_auc_score で LDA の AUC を出し、ロジスティック回帰と比べる

# 解答例③：LDA の AUC
print('ロジスティック回帰:', round(roc_auc_score(y_te, prob), 3))
print('LDA              :', round(roc_auc_score(y_te, prob_lda), 3))


LDAのAUCは0.940で、ロジスティック回帰（0.951）とほとんど同じです。考え方の違う2つの方法が、同じくらいよく分けられました。AUCは、心臓病の人と健康な人の組のうち、心臓病の人の確率のほうが高い組の割合です。心臓病の28人と健康な33人の組は924組あり、差の0.011は、そのうち約10組の違いにすぎません。61人のテストでは、この差は運の範囲と考えられます。1人の確率が少し動くだけで、その人が入っている組（最大33組）が入れ替わるので、約10組の差は、テストの顔ぶれが少し違えば逆になりうるからです。


---


## 7. 答えを出す：見逃しを数で言う


混同行列、適合率と再現率、AUCで、判定の良さを測れるようになりました。最初の問いに戻ります。健診センターの医師として、この判定の結果を、数で説明します。


### お手本：判定の結果を文にする

- 役割：混同行列の数から、見逃しと誤検出をf-stringで文にする
- 入力：`tp`・`pred`・`y_te`
- 出力：報告の文


In [ ]:
fn = ((pred == 0) & (y_te == 1)).sum()    # 見逃し（TODO①と同じ）
n_sick = tp + fn                          # 本当に心臓病の人
print(f'心臓病の{n_sick}人のうち{tp}人を見つけ、{fn}人を見逃した'
      f'（再現率 {tp / n_sick:.2f}）')


### TODO④：誤検出も文にする

お手本をまねて、「健康な○人のうち○人を、心臓病と判定した（誤検出）」の文を表示してください。健康な人の数は`tn + fp`です。


In [ ]:
# TODO: 健康な人のうち、誤って心臓病と判定した人数を文にする

# 解答例④：誤検出の文
fp = ((pred == 1) & (y_te == 0)).sum()
n_ok = tn + fp                            # 本当は健康な人
print(f'健康な{n_ok}人のうち{fp}人を、心臓病と判定した（誤検出）')


医師としての答えです。テストの61人では、しきい値0.5で判定すると、心臓病の28人のうち26人を見つけ、見逃したのは2人でした。健康な33人のうち、誤って精密検査に回ったのは6人です。

まだ言えないことが、3つあります。

- しきい値はどこがよいか：見逃し2人を0人にするには、しきい値をどこまで下げればよいか
- 数の顔をした区分：胸の痛みの型（`cp`）などは、区分なのに、数のまま入れている
- 確率は信じてよいか：0.3という確率が、本当に「30%」なのか（発展）


---


## 目標に答えられたか


- 今日の目標は「心臓病の疑いがある人を見つけ出し、何人を見逃すかを数で言う」ことでした。7節の答えで、見逃しと誤検出は何人でしたか？
- 2節で、線形回帰で0/1を当てると何が困りましたか？ ロジスティック回帰はそれをどう解決しましたか？
- TODO①で、見逃し（FN）を数えるのに、どんな条件を重ねましたか？
- 適合率と再現率のうち、健診で重く見るべきはどちらですか？ その理由は？


---


## 今日の要点


- 0/1を直線で当てると、0〜1の外にはみ出す。ロジスティック回帰はシグモイドで確率にする
- 確率をしきい値と比べると判定になる。正解率は`(pred == y_te).mean()`、Trueの割合
- 混同行列の4つのマスは、条件を`&`で重ねたTrue/Falseの数。見逃し（FN）と誤検出（FP）は重さが違う
- 適合率は誤検出の少なさ、再現率は見逃しの少なさ。正解率だけで語らない
- ROC曲線は、しきい値を動かしたときの誤検出率と検出率の組。その下の面積AUCが、しきい値によらない判定の良さ。ロジスティック回帰とLDAはほぼ同じ
- Python：配列どうしの比較でTrue/Falseの並びを作り、`.sum()`で数、`.mean()`で割合を出す


---


## 次回へ


今日は、しきい値を0.5に決めて判定し、2人を見逃しました。

次回は、新しい受診者も、同じ手順で判定できるようにします。そして、しきい値を動かして、見逃しと誤検出の釣り合いを見ます。どの確率から、精密検査に回せばよいでしょうか。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

今日は13本の検査値をすべて使いました。今度は、最大心拍数（`thalach`）・心電図の下がり幅（`oldpeak`）・年齢（`age`）の3本だけでロジスティック回帰を立て、テストでのしきい値0.5の判定から、混同行列と再現率を表示してください。

1節の`prep`は、使う列の並びを3つ目の引数`cols`で受け取ります。3本の並び`feat3`を作って、`prep`に渡してください（13本の`feat`は、そのまま残しておきます）。


In [ ]:
feat3 = ['thalach', 'oldpeak', 'age']     # 3本だけ
Z3_tr, Z3_te = prep(tr, tr, feat3), prep(tr, te, feat3)
clf3 = LogisticRegression(max_iter=5000).fit(Z3_tr, y_tr)
pred3 = (clf3.predict_proba(Z3_te)[:, 1] > 0.5).astype(int)
print(confusion_matrix(y_te, pred3))
tp3 = ((pred3 == 1) & (y_te == 1)).sum()
fn3 = ((pred3 == 0) & (y_te == 1)).sum()
print('見逃し:', fn3, '人')


<details><summary>詰まったら</summary>

`feat3 = ['thalach', 'oldpeak', 'age']`を作り、`Z3_tr, Z3_te = prep(tr, tr, feat3), prep(tr, te, feat3)`で整えます。
あとは2節・3節と同じく`LogisticRegression(max_iter=5000).fit(Z3_tr, y_tr)` → `predict_proba(Z3_te)[:, 1]` → `> 0.5`です。再現率は`tp / (tp + fn)`です。

</details>


### 応用②（判断）

応用①の3本の式の、テストでの再現率はいくつですか。小数第2位まで答えてください（例: 0.57）。


In [ ]:
print('答え:', round(tp3 / (tp3 + fn3), 2))


<details><summary>詰まったら</summary>

4節の数え方で`tp`と`fn`を数え、`round(tp / (tp + fn), 2)`を表示します。

</details>


### 応用③（解釈）

13本の式（再現率0.93、見逃し2人）と応用①の3本の式を比べて、「検査の項目を3つに減らしてよいか」を、健診センターの運営担当者に向けて3行程度で書いてください。見逃しの人数に触れてください。


（模範例）

13本の検査値を使うと、心臓病の28人のうち見逃しは2人（再現率0.93）でしたが、最大心拍数・心電図の下がり幅・年齢の3本だけにすると、見逃しは6人（再現率0.79）に増えました。

3本に減らすと、心臓病の人の5人に1人を「健康」と帰してしまうことになり、健診の目的である「見逃さないこと」が大きく損なわれます。

検査の手間は増えますが、胸の痛みの型や血管の検査など、ほかの項目も残して判定するほうがよいと考えます。


---


## 発展（任意）


### キャリブレーション曲線：その確率は信じてよいか

ここまで、確率0.5で区切ってきました。でも、しきい値の議論は、`predict_proba`の0.3が本当に「30%」であってこそ成り立ちます。モデルの出す確率が、実際に病気だった割合と合っているかをキャリブレーション（較正）と呼びます。

確かめ方は単純です。予測確率で患者を区間に分け、区間ごとに「実際に病気だった割合」を数えて、予測確率の平均と並べます。点が対角線に乗れば、その確率は信じてよい、ということです。


In [ ]:
from sklearn.calibration import CalibrationDisplay

clf = LogisticRegression(max_iter=5000).fit(Z_tr, y_tr)
CalibrationDisplay.from_estimator(clf, Z_te, y_te, n_bins=5)
plt.title('キャリブレーション曲線（テスト61人）')
plt.show()


In [ ]:
# 区間ごとの人数も見る（61人なので、中央の区間は人数が少ない）
prob = clf.predict_proba(Z_te)[:, 1]
grp = pd.cut(prob, bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
t = pd.DataFrame({'予測確率': prob, '実際': y_te})
g = t.groupby(grp, observed=True)
print(g.agg(人数=('実際', 'size'), 予測の平均=('予測確率', 'mean'),
            実際の割合=('実際', 'mean')).round(2))


読み方：横軸が予測確率の平均、縦軸がその区間で実際に病気だった割合です。点線の対角線に近いほど、確率が「言葉どおり」だと読めます。

両端の区間はよく合っています。中央の区間は人数が少ないので、点が大きく揺れます。数人の違いで割合が大きく変わる区間で「確率が歪んでいる」と言い切るのは早計です。曲線は人数の多い区間を信じ、少ない区間は割り引いて読みます。

試すなら、`LinearDiscriminantAnalysis()`でも同じ図を描いて、比べてみましょう。
